# CH04 · Laboratoire NB02 — Pairs trading chronologique

**Question directrice :** une relation statistique peut-elle être traduite en décisions, ordres et P&L net sans fuite temporelle ?

**Convention explicite :** signal à la clôture t ; position cible exécutée à l'ouverture t+1 ; rendement sur l'intervalle **ouverture t+1 → ouverture t+2**. Le backtest est une **approximation pédagogique** : poids notionnels constants sur chaque intervalle, coûts de turnover entre poids cibles (dérive et coûts de marge simplifiés), gap d'ouverture synthétique, frais paramétriques.

**Limites :** aucune donnée réelle, aucune hypothèse de capacité ou d'emprunt garantie. Les résultats ne sont pas une recommandation de trading.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, coint

SEED = 20261109
rng = np.random.default_rng(SEED)
n = 700
idx = pd.bdate_range('2024-01-02', periods=n)
innovation = rng.normal(0, .007, n)
x = 4.5 + np.cumsum(innovation)
e = np.zeros(n)
for t in range(1, n):
    e[t] = .66 * e[t-1] + rng.normal(0, .012)
trend_other = np.cumsum(rng.normal(0, .006, n))
y_good = -.52 + 1.2 * x + e
y_other = -.52 + 1.2 * x + trend_other
y_break = -.52 + 1.2 * x + e + (np.arange(n) >= 470) * .14
# Open(t) = clôture(t-1) * exp(gap(t)), sans futur; le gap est synthétique.
opens = {}
for k, logprice in [('A',x),('B_C',y_good),('B_N',y_other),('B_R',y_break)]:
    close = np.exp(logprice)
    g = rng.normal(0, .002, n)
    opening = np.r_[close[0], close[:-1] * np.exp(g[1:])]
    opens[k] = opening

df = pd.DataFrame({
  'log_A':x,'log_B_C':y_good,'log_B_N':y_other,'log_B_R':y_break,
  'close_A':np.exp(x),'close_B_C':np.exp(y_good),'close_B_N':np.exp(y_other),'close_B_R':np.exp(y_break),
  'open_A':opens['A'],'open_B_C':opens['B_C'],'open_B_N':opens['B_N'],'open_B_R':opens['B_R']
}, index=idx)
FORM=400; VAL=550
print('Dataset synthétique :', df.shape, '| graine :', SEED)
print('Formation :',df.index[0].date(), '→',df.index[FORM-1].date())
print('Validation :',df.index[FORM].date(), '→',df.index[VAL-1].date())
print('Test :',df.index[VAL].date(), '→',df.index[-1].date())

## 1. Calibrer UNE FOIS sur la période de formation

In [ ]:
train=df.iloc[:FORM]
xtr=train['log_A']; ytr=train['log_B_C']
fit=sm.OLS(ytr,sm.add_constant(xtr)).fit()
alpha,beta=fit.params.iloc[0],fit.params.iloc[1]
print('alpha =',round(alpha,5),'| beta =',round(beta,5))
# Seuils décidés pour l'expérience, sans optimisation sur la validation ni le test.
ENTRY=1.7; EXIT=.45; STOP=4.5; MAX_DUR=25; ROLL=60; COST_BPS=5.
assert beta>0

## 2. Signaux avec statistiques historiques, sans observation future

In [ ]:
def decide_one(z,q,age):
    if q==0:
        return (1,0) if z<-ENTRY else ((-1,0) if z>ENTRY else (0,0))
    if abs(z)<EXIT or abs(z)>STOP or age>=MAX_DUR:
        return 0,0
    return q,age+1

s=(df['log_B_C']-alpha-beta*df['log_A']).to_numpy()
qprev=0; age=0; history=[]
for t in range(FORM, len(df)-2):
    # Référence connue à la clôture t-1 : observations t-ROLL...t-1.
    past=s[t-ROLL:t]
    mu=past.mean();sigma=past.std(ddof=1)
    z=(s[t]-mu)/sigma if sigma>1e-12 else 0.
    q,age=decide_one(z,qprev,age)
    wB=q/(1+abs(beta));wA=-q*beta/(1+abs(beta))
    pB1=df['open_B_C'].iloc[t+1]; pB2=df['open_B_C'].iloc[t+2]
    pA1=df['open_A'].iloc[t+1]; pA2=df['open_A'].iloc[t+2]
    rB=pB2/pB1-1.;rA=pA2/pA1-1.
    prevB=qprev/(1+abs(beta));prevA=-qprev*beta/(1+abs(beta))
    turnover=abs(wB-prevB)+abs(wA-prevA) # simplification : ignore la dérive des poids avant ordre
    gross=wB*rB+wA*rA
    net=gross-COST_BPS/10000*turnover
    history.append({'signal_cloture':df.index[t], 'execution_ouverture':df.index[t+1],
                    'fin_intervalle':df.index[t+2],'z':z,'q':q,'wB':wB,'wA':wA,
                    'gross':gross,'turnover':turnover,'cost':COST_BPS/10000*turnover,'net':net})
    qprev=q
journal=pd.DataFrame(history).set_index('signal_cloture')
display(journal.head(7).round(4))

**Contrôle de causalité :** le signal d’une ligne utilise t et le passé ; la position issue de ce signal ne reçoit que les rendements suivant l’ouverture t+1. Quelle variable du journal prouve cette chronologie ?

## 3. Évaluation et sensibilité aux coûts

In [ ]:
from math import prod
journal['richesse_brute']=(1+journal['gross']).cumprod()
journal['richesse_nette']=(1+journal['net']).cumprod()
fig,ax=plt.subplots(figsize=(11,4))
ax.plot(journal.index,journal['richesse_brute'],label='Avant coûts')
ax.plot(journal.index,journal['richesse_nette'],label='Après coûts')
ax.axvline(df.index[VAL],color='black',ls='--',label='Début test final')
ax.set(title='Backtest synthétique : référence de richesse 1',ylabel='Richesse approximative');ax.legend();plt.show()

for label,block in [('validation',journal[journal.index<df.index[VAL]]),
                    ('test',journal[journal.index>=df.index[VAL]])]:
    wr=(1+block['net']).prod()-1
    print(label, '| périodes',len(block),'| rendement net',round(wr,4),
          '| turnover',round(block['turnover'].sum(),2), '| signaux actifs',int((block.q!=0).sum()))

## 4. Comparer la performance nette selon les hypothèses de coûts

In [ ]:
for c in [0,5,15,50]:
    r=journal['gross']-(c/10000)*journal['turnover']
    print(f'{c:2d} bps/unité : richesse finale = {(1+r).prod():.4f}')

### Livrable NB02

Fournir une fiche de décision comprenant : heure du signal, cours d’exécution, règle de sortie, coûts testés, comparaison validation/test, drawdown et raisons de suspendre la paire.